# Multimodal RecSys Agent — Kaggle Quickstart
**No API keys required for Phase 1.**

Run order:
1. Setup
2. Data
3. Preprocess
4. Train Mult-VAE
5. Train Two-Tower
6. Build Qdrant Index
7. RecSys Eval
8. (Optional) Agent with Ollama

In [1]:
# ── Cell 1: GPU Check ────────────────────────────────────────────────────
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

GPU available: True
GPU: Tesla T4
VRAM: 15.6GB


In [2]:
# ── Cell 2: Clone Repo ───────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/MadhaviSG/multimodal-recsys-agent.git'
WORK_DIR = '/kaggle/working/multimodal-recsys-agent'

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

Cloning into '/kaggle/working/multimodal-recsys-agent'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (137/137), done.
Receiving objects: 100% (161/161), 111.91 KiB | 3.39 MiB/s, done.
remote: Total 161 (delta 61), reused 75 (delta 9), pack-reused 0 (from 0)
Resolving deltas: 100% (61/61), done.
Working directory: /kaggle/working/multimodal-recsys-agent


In [3]:
# ── Cell 3: Install Dependencies ─────────────────────────────────────────
!pip install -q \
    qdrant-client \
    rank-bm25 \
    lightgbm \
    diffusers \
    accelerate \
    python-dotenv \
    scipy

# FlagEmbedding separately (larger)
!pip install -q FlagEmbedding

print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 7.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 19.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.7 MB/s eta 0:00:00
Dependencies installed.


In [34]:
!python scripts/download_data.py

ml-25m.zip: 100%|████████████████████████████| 262M/262M [00:02<00:00, 90.3MB/s]
Extracting...
✓ MovieLens 25M extracted to data/raw/ml-25m/
  Files: ['tags.csv', 'ratings.csv', 'README.txt', 'links.csv', 'movies.csv', 'genome-tags.csv', 'genome-scores.csv']


In [36]:
# Verify
import pandas as pd
ratings = pd.read_csv('data/raw/ml-25m/ratings.csv')
print(f'\nRatings loaded: {len(ratings):,} rows')
print(ratings.head(3))


Ratings loaded: 25,000,095 rows
   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828


In [37]:
# ── Cell 5: Preprocess ───────────────────────────────────────────────────
!python scripts/preprocess.py

# Verify outputs
import scipy.sparse as sp
train = sp.load_npz('data/processed/train.npz')
print(f'Train matrix: {train.shape}, nnz={train.nnz:,}')

Loading ratings.csv...
  Loaded 25,000,095 interactions, 162,541 users, 59,047 items

K-core filtering (min_user=20, min_item=5)
  Iteration 1: removed 54,225 interactions (162,541 users, 32,720 items remain)
  Iteration 2: removed 480 interactions (162,516 users, 32,711 items remain)
  Iteration 3: removed 0 interactions (162,516 users, 32,711 items remain)
  Converged after 3 iterations.

Temporal split:
  Train: 19,956,312 interactions (1995-01-09 → 2016-06-18)
  Val:   2,494,539 interactions (2016-06-18 → 2017-12-29)
  Test:  2,494,539 interactions (2017-12-29 → 2019-11-21)

ID maps built from train set:
  Users: 137,723
  Items: 26,224

Building sparse matrices...
  Train:
  Matrix shape: (137723, 26224), density: 0.5526%
  Val:
  Matrix shape: (137723, 26224), density: 0.0103%
  Test:
  Matrix shape: (137723, 26224), density: 0.0065%

Saving to data/processed/...

✓ Done in 28.0s
{
  "n_users": 137723,
  "n_items": 26224,
  "n_train": 19956312,
  "n_val": 370771,
  "n_test": 2361

In [83]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

In [84]:
# ── Cell 6: Git Pull + Config Setup ────────────────────────────────────
# Pull first — git pull would overwrite any config edits made before it.
import os, yaml

os.chdir('/kaggle/working/multimodal-recsys-agent')
!git pull origin main

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

config['training']['log_wandb'] = True

# T4 VRAM: reduce batch size
config['two_tower']['batch_size'] = 256

# n_samples = 2 × n_users (275K) — 2 expected updates/user/epoch, 86.5% coverage
config['two_tower']['n_samples'] = 275000

# lr=0.001: 1074 steps × 0.001 = 1.07 displacement/epoch (safe for embed_norm≈8)
config['two_tower']['lr'] = 0.001

# weight_decay=1e-4: 0.01 was same magnitude as BPR gradient, drowning signal
config['two_tower']['weight_decay'] = 0.0001

config['two_tower']['n_epochs'] = 30

with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config written:')
print(f'  n_samples   : {config["two_tower"]["n_samples"]:,}')
print(f'  lr          : {config["two_tower"]["lr"]}')
print(f'  weight_decay: {config["two_tower"]["weight_decay"]}')
print(f'  n_epochs    : {config["two_tower"]["n_epochs"]}')
print(f'  batch_size  : {config["two_tower"]["batch_size"]}')


Config updated.
  batch_size: 256
  n_epochs: 10
  log_wandb: True


In [ ]:
# ── Sanity Check: 5 epochs, confirm loss decreases ───────────────────────
# Run this BEFORE the full 30-epoch run.
# Pass criteria: epoch 5 avg_loss < epoch 1 avg_loss (any decrease = learning).
import yaml

with open('configs/config.yaml') as f:
    c = yaml.safe_load(f)
c['two_tower']['n_epochs'] = 5
with open('configs/config.yaml', 'w') as f:
    yaml.dump(c, f, default_flow_style=False)

print(f'[SANITY] Running 5 epochs — n_samples={c["two_tower"]["n_samples"]:,}, '
      f'lr={c["two_tower"]["lr"]}, weight_decay={c["two_tower"]["weight_decay"]}')
!python scripts/train_two_tower.py --config configs/config.yaml

# After this cell: check that epoch 5 loss < epoch 1 loss.
# If yes → run Cell 10 (full 30-epoch training) with n_epochs restored to 30.
# If loss still flat → run the iALS fallback cell below.


In [95]:
import os
os.environ['PYTHONPATH'] = '/kaggle/working/multimodal-recsys-agent'

In [ ]:
# ── Cell 7: Train Mult-VAE ───────────────────────────────────────────────
# ~45 min for 10 epochs on T4
# ~2.5 hrs for full 50 epochs

!python scripts/train_mult_vae.py --config configs/config.yaml

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mgulavan (mgulavan-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ setting up run um5hpcx4 (0.3s)
wandb: ⣷ setting up run um5hpcx4 (0.3s)
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/multimodal-recsys-agent/wandb/run-20260405_043341-um5hpcx4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mult_vae_1775363621
wandb: ⭐️ View project at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent
wandb: 🚀 View run at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent/runs/um5hpcx4
Loading processed data...
  Train matrix: (137723, 26224), nnz=19,956,312
  Val matrix:   (137723, 262

In [112]:
# ── DEPRECATED — temperature already set to 0.2 in two_tower.py ──────────
# This cell is a no-op. Kept for reference only.
# content = open('src/recsys/models/two_tower.py').read()
# content = content.replace('self.temperature = 0.07', 'self.temperature = 0.2')
# open('src/recsys/models/two_tower.py', 'w').write(content)
print('Skipped: temperature already 0.2 in repo.')


Ready to train


In [113]:
import torch
from src.recsys.models.two_tower import TwoTowerModel

model = TwoTowerModel(num_users=10, num_items=10, item_feature_dim=21)
print(model)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
# Should show ONLY user_embed + item_embed — no MLP layers

TwoTowerModel(
  (user_embed): Embedding(10, 64)
  (item_embed): Embedding(10, 64)
)
Params: 1,280


In [125]:
# ── DEPRECATED — lr is set in Cell 6 (config setup) above ───────────────
# Remove this cell or it will silently override the config written in Cell 6.
import yaml
with open('configs/config.yaml') as f:
    c = yaml.safe_load(f)
print('Current lr:', c['two_tower']['lr'])   # verify, don't override


lr: 0.003


In [126]:
# ── Cell 8: Train Two-Tower — Full Run ──────────────────────────────────
# Restore n_epochs=30, then train.
import yaml
with open('configs/config.yaml') as f:
    c = yaml.safe_load(f)
c['two_tower']['n_epochs'] = 30
with open('configs/config.yaml', 'w') as f:
    yaml.dump(c, f, default_flow_style=False)
print(f'Training {c["two_tower"]["n_epochs"]} epochs, '
      f'n_samples={c["two_tower"]["n_samples"]:,}, lr={c["two_tower"]["lr"]}')
!python scripts/train_two_tower.py --config configs/config.yaml


From https://github.com/MadhaviSG/multimodal-recsys-agent
 * branch            main       -> FETCH_HEAD
Already up to date.
Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mgulavan (mgulavan-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/multimodal-recsys-agent/wandb/run-20260405_141733-e9rn9lkc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run two_tower_1775398653
wandb: ⭐️ View project at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent
wandb: 🚀 View run at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent/runs/e9rn9lkc
Loading processed data...
  137,723 users, 26,224 items
Item features: (26224, 21), genre spar

In [128]:
!git pull origin main

From https://github.com/MadhaviSG/multimodal-recsys-agent
 * branch            main       -> FETCH_HEAD
Already up to date.


In [129]:
from importlib import reload
import src.recsys.models.two_tower as tt
reload(tt)
from src.recsys.models.two_tower import TwoTowerModel
print([m for m in dir(TwoTowerModel) if not m.startswith('_')])
# Should show: bpr_loss, forward, get_item_embedding, get_user_embedding, in_batch_loss

['T_destination', 'add_module', 'apply', 'bfloat16', 'bpr_loss', 'buffers', 'call_super_init', 'children', 'compile', 'cpu', 'cuda', 'double', 'dump_patches', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_item_embedding', 'get_parameter', 'get_submodule', 'get_user_embedding', 'half', 'in_batch_loss', 'ipu', 'load_state_dict', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_submodule', 'share_memory', 'state_dict', 'to', 'to_empty', 'train', 'type', 'xpu', 'zero_grad']


In [130]:
import torch
import scipy.sparse as sp
from src.recsys.models.two_tower import TwoTowerModel

train_matrix = sp.load_npz('data/processed/train.npz')
n_users, n_items = train_matrix.shape

model = TwoTowerModel(num_users=n_users, num_items=n_items).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

user_ids = torch.tensor([0, 1, 2, 3, 4], dtype=torch.long).cuda()
pos_ids  = torch.tensor([100, 200, 300, 400, 500], dtype=torch.long).cuda()
neg_ids  = torch.tensor([1000, 2000, 3000, 4000, 5000], dtype=torch.long).cuda()

for step in range(20):
    optimizer.zero_grad()
    loss = model.bpr_loss(user_ids, pos_ids, neg_ids)
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f"Step {step}: loss={loss.item():.6f}")

Step 0: loss=0.618356
Step 5: loss=0.598997
Step 10: loss=0.580188
Step 15: loss=0.561975


In [131]:
!python scripts/train_two_tower.py --config configs/config.yaml

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mgulavan (mgulavan-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/multimodal-recsys-agent/wandb/run-20260405_142337-k7ngjwhj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run two_tower_1775399017
wandb: ⭐️ View project at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent
wandb: 🚀 View run at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent/runs/k7ngjwhj
Loading processed data...
  137,723 users, 26,224 items
Item features: (26224, 21), genre sparsity: 89.91%
Building user-item index...
BPRDataset: 2,000,000 samples/epoch, 137,723 active users

Model: 10,492,608 parame

In [132]:
# ── DEPRECATED — n_samples is now set in Cell 6 (config setup) ──────────
# Verify only:
import yaml
with open('configs/config.yaml') as f:
    c = yaml.safe_load(f)
print('n_samples:', c['two_tower'].get('n_samples', 'NOT SET'))
print('lr:', c['two_tower']['lr'])
print('n_epochs:', c['two_tower']['n_epochs'])


n_samples: 200000


In [ ]:
# Cell 1 — restart kernel first
import os
os.kill(os.getpid(), 9)

In [1]:
import yaml

# Verify config first
with open('configs/config.yaml') as f:
    c = yaml.safe_load(f)
print("n_samples:", c['two_tower'].get('n_samples', 'NOT SET'))
print("lr:", c['two_tower']['lr'])
print("n_epochs:", c['two_tower']['n_epochs'])

FileNotFoundError: [Errno 2] No such file or directory: 'configs/config.yaml'

In [100]:
# ── Cell 8: Train Two-Tower ──────────────────────────────────────────────
# ~30 min for 10 epochs on T4

# !python scripts/train_two_tower.py --config configs/config.yaml

Device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mgulavan (mgulavan-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/multimodal-recsys-agent/wandb/run-20260405_051151-2dr3gxc8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run two_tower_1775365911
wandb: ⭐️ View project at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent
wandb: 🚀 View run at https://wandb.ai/mgulavan-carnegie-mellon-university/multimodal-recsys-agent/runs/2dr3gxc8
Loading processed data...
  137,723 users, 26,224 items
Item features: (26224, 21), genre sparsity: 89.91%
Building user-item index...
InteractionPairDataset: 2,000,000 samples/epoc

In [41]:
# ── Cell 9: Build Qdrant Index ───────────────────────────────────────────
# Uses in-memory Qdrant — no Docker needed

import subprocess, sys

# Patch config to use in-memory Qdrant
with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)
config['retrieval']['qdrant_url'] = ':memory:'
with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f)

!python scripts/build_index.py --config configs/config.yaml --recreate

Building Qdrant index...
Loading embeddings from checkpoints/item_embeddings.npy...
Traceback (most recent call last):
  File "/kaggle/working/multimodal-recsys-agent/scripts/build_index.py", line 371, in <module>
    main(config, recreate=args.recreate)
  File "/kaggle/working/multimodal-recsys-agent/scripts/build_index.py", line 304, in main
    embeddings = np.load(emb_path)
                 ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 455, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/item_embeddings.npy'


In [117]:
!git restore src/recsys/models/two_tower.py
!git pull origin main

From https://github.com/MadhaviSG/multimodal-recsys-agent
 * branch            main       -> FETCH_HEAD
Updating 4ebf001..8d05864
Fast-forward
 scripts/train_two_tower.py     | 65 ++++++++++++++++++++++--------------------
 src/recsys/models/two_tower.py | 19 +++++++++++-
 2 files changed, 52 insertions(+), 32 deletions(-)


In [42]:
# ── Cell 10: RecSys Eval (No Keys Needed) ───────────────────────────────
import sys
sys.path.insert(0, '.')

import scipy.sparse as sp
import numpy as np
from src.eval.recsys.metrics import ndcg_at_k, recall_at_k, coverage, novelty

print('Running RecSys metrics...')
# Quick manual eval on 100 val users
# (full eval harness needs OpenAI key for agent eval — skip for now)

train_mat = sp.load_npz('data/processed/train.npz')
val_mat = sp.load_npz('data/processed/val.npz')

import torch
from src.recsys.models.mult_vae import MultVAE
import json

with open('data/processed/item2idx.json') as f:
    item2idx = json.load(f)

n_users, n_items = train_mat.shape
device = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load('checkpoints/mult_vae_best.pt', map_location=device)
model = MultVAE(
    num_items=n_items,
    hidden_dims=ckpt['config']['recsys']['hidden_dims'],
    latent_dim=ckpt['config']['recsys']['latent_dim'],
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

K = 10
rng = np.random.RandomState(42)
eval_users = rng.choice(n_users, size=100, replace=False)

ndcg_scores, recall_scores = [], []

with torch.no_grad():
    for uid in eval_users:
        train_row = train_mat[uid].toarray().squeeze(0)
        x = torch.tensor(train_row, dtype=torch.float32).unsqueeze(0).to(device)
        recon, _, _ = model(x)
        scores = recon.squeeze(0).cpu().numpy()
        scores[train_row > 0] = -np.inf  # mask seen items

        val_row = val_mat[uid].toarray().squeeze(0)
        relevant = set(np.where(val_row > 0)[0])
        if not relevant:
            continue

        top_k = np.argsort(scores)[::-1][:K]
        ndcg_scores.append(ndcg_at_k(list(top_k), relevant, K))
        recall_scores.append(recall_at_k(list(top_k), relevant, K))

print(f'\n=== Mult-VAE Eval Results (100 users) ===')
print(f'NDCG@{K}:   {np.mean(ndcg_scores):.4f}')
print(f'Recall@{K}: {np.mean(recall_scores):.4f}')
print(f'(Baseline NDCG@10 from paper: ~0.44)')

ModuleNotFoundError: No module named 'src.eval.recsys'

In [ ]:
# ── Cell 11: Save Checkpoints ────────────────────────────────────────────
# Save to /kaggle/working so you can download them
import shutil

shutil.copytree('checkpoints', '/kaggle/working/checkpoints', dirs_exist_ok=True)
shutil.copytree('data/processed', '/kaggle/working/data_processed', dirs_exist_ok=True)

print('Saved to Kaggle output:')
for f in os.listdir('/kaggle/working/checkpoints'):
    size = os.path.getsize(f'/kaggle/working/checkpoints/{f}') / 1e6
    print(f'  {f}: {size:.1f}MB')

In [ ]:
# ── Cell 12 (Optional): Run Agent With Ollama ────────────────────────────
# Only run this if you want to test the agent layer without OpenAI

# Install and start Ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

# Pull small model (fits in 15GB VRAM alongside RecSys models)
!ollama pull phi3:mini

# Update config to use Ollama
with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)
config['agent']['llm_model'] = 'phi3:mini'
with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f)

# Test query
!python scripts/run_agent.py --query 'Recommend me psychological thrillers'

In [ ]:
# ── iALS Fallback — Guaranteed Convergence ───────────────────────────────
# Run this ONLY if BPR still does not converge after the sanity check.
# Uses the `implicit` library (ALS on sparse interaction matrix).
# Output: item_embeddings.npy (26224, 64) + two_tower_best.pt
# Compatible with the rest of the pipeline (Qdrant indexing, eval).
#
# Reference: used in production at Spotify, Netflix.

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
import json
from pathlib import Path

try:
    import implicit
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'implicit'])
    import implicit

# Load interaction matrix
train_matrix = sp.load_npz('data/processed/train.npz')
n_users, n_items = train_matrix.shape
print(f'Matrix: {n_users:,} users × {n_items:,} items, nnz={train_matrix.nnz:,}')

# Binarize (implicit feedback) and transpose to (items × users)
conf = train_matrix.astype(np.float32)
conf.data[:] = 1.0
item_user = conf.T.tocsr()

# Train ALS
model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=30,
    regularization=0.01,
    random_state=42,
)
print('Fitting ALS...')
model.fit(item_user)

# Extract factors
item_factors = np.array(model.item_factors)  # (n_items, 64)
user_factors = np.array(model.user_factors)  # (n_users, 64)
print(f'item_factors: {item_factors.shape}, user_factors: {user_factors.shape}')

# Normalize (matches TwoTowerModel.get_item_embedding which applies F.normalize)
item_embs = F.normalize(torch.tensor(item_factors, dtype=torch.float32), dim=-1).numpy()
user_embs = F.normalize(torch.tensor(user_factors, dtype=torch.float32), dim=-1).numpy()

# Save item embeddings for Qdrant indexing
Path('checkpoints').mkdir(exist_ok=True)
np.save('checkpoints/item_embeddings.npy', item_embs)
print(f'Saved item_embeddings.npy: {item_embs.shape}')  # (26224, 64)

# Save as two_tower_best.pt — state_dict keys match TwoTowerModel
# (user_embed.weight, item_embed.weight).  F.normalize applied at query time.
torch.save({
    'epoch': 30,
    'model_state_dict': {
        'user_embed.weight': torch.tensor(user_factors, dtype=torch.float32),
        'item_embed.weight': torch.tensor(item_factors, dtype=torch.float32),
    },
    'recall': 0.0,
    'config': {},
    'n_users': n_users,
    'n_items': n_items,
    'num_items': n_items,
    'item_feature_dim': 21,
}, 'checkpoints/two_tower_best.pt')
print('Saved two_tower_best.pt')

# Quick Recall@10 sanity check on 200 random users
rng = np.random.RandomState(42)
val_matrix = sp.load_npz('data/processed/val.npz')
val_users = rng.choice(n_users, size=200, replace=False)
item_embs_t = torch.tensor(item_embs)
user_embs_t = torch.tensor(user_embs)

recalls = []
for uid in val_users:
    scores = (user_embs_t[uid] @ item_embs_t.T).numpy()
    train_items = train_matrix[uid].indices
    scores[train_items] = -np.inf
    val_items = set(val_matrix[uid].indices)
    if not val_items:
        continue
    top10 = np.argsort(scores)[::-1][:10]
    recalls.append(sum(1 for i in top10 if i in val_items) / len(val_items))

print(f'ALS Recall@10 (200 users): {np.mean(recalls):.4f}')
print('Pipeline ready for Qdrant indexing.')
